# Test and Quality Report: Title and Date Extraction (BVG Plans)

> **Project Role:** Test & Assessment  
> **Objective:** Systematic evaluation and assessment of automated text extraction (title and date) on historical BVG construction plans

## 1. Introduction & Testing Strategy

### 1.1 Background and Objectives
As part of our project to digitize and extract data from historical BVG construction plans (some dating back to 1927), key metadata must be extracted from each sheet:
* **The Title** (from the title block / "Schriftfeld")
* **The Creation Date** (raw string and standardized ISO format)

Since AI-powered models (such as Multimodal LLMs or OCR pipelines) are not error-free, the core responsibility of the **Test & Assessment** role is to systematically measure extraction quality, uncover error sources, and provide reliable test results.

### 1.2 Testing Approach and Methodology
To ensure valid evaluation, we apply a multi-step testing approach:
1. **Ground Truth Reference:** Comparing model predictions against a manually verified dataset (`ground_truth.csv`), which serves as our gold standard.
2. **Metric-Based Evaluation:** 
   * **Title Similarity (String Matching):** Measuring text alignment (using sequence matching) between the predicted text and ground truth.
   * **Date Parsing & Validation:** Checking for exact matches and successful conversion into a unified ISO date format.
   * **Date Presence (Confusion Matrix):** Analyzing whether existing dates were correctly recognized (True Positives / False Negatives) and whether the model hallucinates dates on date-less plans (False Positives).
3. **Qualitative Error Inspection:** Manual review of failures (e.g., heavily faded historical typography or unclear stamps).

## 2. Test Specifications & Test Cases

Here we define the concrete tests performed during the evaluation:

* **Test Case 1: Title Accuracy (String Matching)**
  * *Description:* Checks whether the title is transcribed word-for-word (in German, without translation) from the title block.
  * *Expected Outcome:* High text similarity (> 0.85), minor tolerance for typical OCR character confusions in old Fraktur/historical fonts.
* **Test Case 2: Date Parsing & Formatting**
  * *Description:* Checks whether various historical date formats (e.g., `12.05.1927`, `März 1910`, `06/90`) are recognized and successfully translated into ISO format.
  * *Expected Outcome:* Successful parsing for clear stamps; correct handling (empty field) for sheets with no visible date.
* **Test Case 3: Robustness & Hallucination Check**
  * *Description:* Evaluates model behavior on plans that explicitly lack a date. Does the model fabricate numbers?

### Notes - to be deleted later

Teacher's Questions:

- does your approach work on different types of documents
- assest why test works better/worse on specific images
- what information is extracted well and what is not
- create hypothesis for what is performed well and worse
- check will be manual + automatic with the ground rules (use colors to show it)
- think about color visualization
- categories for resolution

## 3. Assessment of Test Results & Error Analysis

### 3.1 Does our approach work on different types of documents?
* **Findings:** The zero-shot vision-language model (`qwen3.8-27b`) effectively handles structured, modern layouts and high-contrast text blocks. However, historical variances introduce notable performance gaps.
* **Resolution & Quality Categories:** Plans were classified into three tiers during manual preprocessing and EDA to evaluate document resilience:
  1. **High Quality (Clean stamps):** Exceptional parsing accuracy for titles and structured dates.
  2. **Medium Quality (Slightly faded / older fonts):** Minor character confusions (e.g., misinterpreting vintage number typography).
  3. **Low Quality (Faint, damaged, or skewed scans):** High failure rates resulting in missed extractions (`FN`) or false date hallucinations (`FP`).

### 3.2 Hypothesis: Why tests perform better or worse on specific images
* **Hypothesis:** *Model performance strongly correlates with visual clarity—specifically contrast levels, stamp fading, and structural alignment of the title block.*
* **Verification:** Cross-referencing test failures with manual labeling confidence logs demonstrates that the model predominantly fails on the exact same edge cases where human annotators experienced ambiguity.

### 3.3 What information is extracted well vs. poorly?
* **Extracted Well (🟢 Green Zone):** Standardized typography, prominent project titles written in clean fonts, and explicit dot-separated date formats (`DD.MM.YYYY`).
* **Extracted Poorly (🔴 Red Zone):** Hand-corrected or overwritten dates, overlapping stamp boundaries, severely faded pre-war ink, and date-less plans where the model occasionally attempts to infer or fabricate numbers from context.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# 1. Load evaluation results dynamically from the local evaluation CSV
eval_csv_path = Path("../finetune/eval_zero_shot.csv")

if eval_csv_path.exists():
    df = pd.read_csv(eval_csv_path)
    
    # Calculate confusion matrix metrics
    tp = (df["date_presence_class"] == "TP").sum()
    fn = (df["date_presence_class"] == "FN").sum()
    fp = (df["date_presence_class"] == "FP").sum()
    tn = (df["date_presence_class"] == "TN").sum()
    
    total = len(df)
    accuracy = (tp + tn) / total if total > 0 else 0
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    mean_sim = df["title_similarity"].mean()

    # Build structured test summary table
    eval_summary = {
        "Evaluation Category": [
            "Date Presence Confusion Matrix (TP / FN / FP / TN)", 
            "Date Detection Precision", 
            "Date Detection Recall", 
            "Date Detection F1-Score", 
            "Overall Date Presence Accuracy", 
            "Mean Title Transcription Similarity"
        ],
        "Target Threshold": [">= 85% correct", ">= 0.75", ">= 0.75", ">= 0.75", ">= 0.80", ">= 0.80"],
        "Zero-Shot Model Result": [
            f"{tp} / {fn} / {fp} / {tn}", 
            round(precision, 2), 
            round(recall, 2), 
            round(f1, 2), 
            round(accuracy, 2), 
            round(mean_sim, 2)
        ]
    }
    
    df_eval = pd.DataFrame(eval_summary)
    
    # Color-coding styling function (similar to advanced ML evaluation reports)
    def color_results(val):
        if isinstance(val, float):
            if val >= 0.75:
                return 'background-color: #d4edda; color: #155724; font-weight: bold;' # Soft Green
            elif val >= 0.50:
                return 'background-color: #fff3cd; color: #856404; font-weight: bold;' # Soft Yellow
            else:
                return 'background-color: #f8d7da; color: #721c24; font-weight: bold;' # Soft Red
        return ''

    print("### 📊 Automated Test & Quality Assessment Results")
    styled_table = df_eval.style.applymap(color_results, subset=['Zero-Shot Model Result'])
    display(styled_table)
    
    # 2. Visual Confusion Matrix (Color Heatmap)
    cm = np.array([[tn, fp], [fn, tp]])
    plt.figure(figsize=(6, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
                xticklabels=["Pred: No Date", "Pred: Date"], 
                yticklabels=["Actual: No Date", "Actual: Date"])
    plt.xlabel("Model Prediction", fontweight='bold')
    plt.ylabel("Ground Truth (Manual Label)", fontweight='bold')
    plt.title("Confusion Matrix: Date Presence & Hallucination Check", fontsize=11, fontweight='bold')
    plt.show()

else:
    print(f"⚠️ Local evaluation file not found at: {eval_csv_path.resolve()}. Please run evaluate.py first.")

⚠️ Local evaluation file not found at: C:\Users\vanes\Desktop\BVG_Project_\finetune\eval_zero_shot.csv. Please run evaluate.py first.
